# Proyecto Terminal: Predicción de Cancelación de Clientes (Interconnect)

## Condiciones de la asignación principal

Al operador de telecomunicaciones Interconnect le gustaría poder pronosticar su tasa de cancelación de clientes. Si se descubre que un usuario o usuaria planea irse, se le ofrecerán códigos promocionales y opciones de planes especiales. El equipo de marketing de Interconnect ha recopilado algunos de los datos personales de sus clientes, incluyendo información sobre sus planes y contratos.

**Servicios de Interconnect**

Interconnect proporciona principalmente dos tipos de servicios:

1.	Comunicación por teléfono fijo. El teléfono se puede conectar a varias líneas de manera simultánea.
  
2.	Internet. La red se puede configurar a través de una línea telefónica (DSL, línea de abonado digital) o a través de un cable de fibra óptica.
  
Algunos otros servicios que ofrece la empresa incluyen:

* Seguridad en Internet: software antivirus (ProtecciónDeDispositivo) y un bloqueador de sitios web maliciosos (SeguridadEnLínea).

* Una línea de soporte técnico (SoporteTécnico).
* Almacenamiento de archivos en la nube y backup de datos (BackupOnline).
* Streaming de TV (StreamingTV) y directorio de películas (StreamingPelículas)

La clientela puede elegir entre un pago mensual o firmar un contrato de 1 o 2 años. Puede utilizar varios métodos de pago y recibir una factura electrónica después de una transacción

**Descripción de los datos**

Los datos consisten en archivos obtenidos de diferentes fuentes:

* contract.csv — información del contrato;
* personal.csv — datos personales del cliente;
* internet.csv — información sobre los servicios de Internet;
* phone.csv — información sobre los servicios telefónicos.
  
En cada archivo, la columna customerID (ID de cliente) contiene un código único asignado a cada cliente. La información del contrato es válida a partir del 1 de febrero de 2020.

## Fase 1: Plan de Trabajo y Preguntas Aclaratorias

# 1. Entendimiento del Problema de Negocio e Indicadores de Éxito

El objetivo del proyecto es desarrollar un modelo de aprendizaje automático capaz de estimar la probabilidad de cancelación de los clientes de la empresa de telecomunicaciones Interconnect.

La predicción permitirá identificar clientes con mayor riesgo de abandono para que el equipo de marketing pueda implementar acciones preventivas, como promociones, descuentos u ofertas de planes especiales.

Desde el punto de vista de Machine Learning, el problema se abordará como una clasificación binaria supervisada, donde cada cliente será clasificado según su probabilidad de cancelar o permanecer con la compañía.

El proyecto deberá mantener una relación directa entre:

problema de negocio → análisis de datos → construcción del modelo → evaluación → recomendaciones.

# 1.1. Problema de negocio

Interconnect quiere anticipar qué clientes podrían cancelar sus servicios para que el área de Marketing pueda intervenir con promociones, descuentos o planes especiales antes de perderlos.

Por lo tanto, técnicamente tenemos un problema de:

## Clasificación binaria supervisada

El modelo buscará estimar la probabilidad de que un cliente abandone la empresa:

```text
X → P(Churn = 1)
```

Sin embargo, desde el punto de vista empresarial, no se busca únicamente obtener una predicción de tipo **Sí/No**.

Lo más útil será calcular una probabilidad individual de abandono (*churn*) para cada cliente. Por ejemplo:

| customerID | churn_probability |
|------------|-------------------|
| A1023      | 0.87              |
| B2914      | 0.74              |
| C8812      | 0.13              |

De esta manera, el área de Marketing podrá priorizar sus esfuerzos según el nivel de riesgo de cada cliente:

- Los clientes con una probabilidad alta de abandono recibirán atención prioritaria.
- Los clientes con una probabilidad intermedia podrán ser considerados para campañas preventivas.
- Los clientes con una probabilidad baja podrán mantenerse dentro de las estrategias habituales de fidelización.

El objetivo final es transformar las predicciones del modelo en acciones comerciales concretas que ayuden a reducir la tasa de cancelación y mejorar la retención de clientes.

# 2. Estructura de los datos

**Diccionario de datos**


| Nombre de tabla | Nombre de columna | Significado / Descripción | Importancia para el científico de datos |
|---|---|---|---|
| `contract` | `customerID` | Identificador único de cada cliente. | Clave primaria. Crucial para unir las cuatro tablas sin duplicar registros. |
| `contract` | `BeginDate` | Fecha en la que el cliente inició el contrato. | Permite calcular la antigüedad (`Tenure`), una de las variables más predictivas del *churn*. |
| `contract` | `EndDate` | Fecha de cancelación del servicio (`No` si el cliente sigue activo). | Variable objetivo indirecta. Si contiene una fecha, el cliente canceló (`Churn = 1`). |
| `contract` | `Type` | Tipo de facturación: mes a mes, un año o dos años. | Variable crítica. Los contratos mes a mes suelen presentar tasas de cancelación más altas. |
| `contract` | `PaperlessBilling` | Indica si el cliente utiliza factura electrónica (`Yes`/`No`). | Variable relacionada con el comportamiento digital y, potencialmente, con la adopción tecnológica y la retención. |
| `contract` | `PaymentMethod` | Método de pago utilizado por el cliente: cheque electrónico, cheque por correo, transferencia o tarjeta. | Los pagos automáticos, como tarjeta o transferencia, pueden reducir el *churn* involuntario provocado por tarjetas vencidas. |
| `contract` | `MonthlyCharges` | Monto cobrado mensualmente al cliente. | Variable financiera. Los clientes con cargos elevados y pocos servicios premium podrían buscar alternativas en la competencia. |
| `contract` | `TotalCharges` | Monto total acumulado que el cliente ha pagado. | Permite estimar el valor de vida del cliente (LTV). Requiere limpieza porque inicialmente se encuentra almacenado como texto. |
| `personal` | `customerID` | Identificador único de cada cliente. | Clave foránea para conectar esta tabla con la tabla `contract`. |
| `personal` | `gender` | Género del cliente (`Male`/`Female`). | Variable demográfica. Puede utilizarse para verificar posibles sesgos en las cancelaciones por género. |
| `personal` | `SeniorCitizen` | Indica si el cliente es adulto mayor (`1`/`0`). | Variable demográfica. Los adultos mayores podrían requerir canales de soporte tradicionales o planes con tarifas fijas. |
| `personal` | `Partner` | Indica si el cliente tiene pareja (`Yes`/`No`). | Puede reflejar estabilidad familiar. Las cuentas familiares o de parejas podrían presentar mayor lealtad y menor *churn*. |
| `personal` | `Dependents` | Indica si el cliente tiene dependientes, como hijos (`Yes`/`No`). | Los clientes con familias grandes podrían priorizar la estabilidad del servicio de internet. |
| `internet` | `customerID` | Identificador único de cada cliente. | Permite identificar a los clientes que tienen contratado el servicio de internet. |
| `internet` | `InternetService` | Tipo de tecnología de internet contratada: DSL o fibra óptica. | La fibra óptica suele ser más rápida, pero también más costosa. Los problemas de red pueden incrementar el *churn*. |
| `internet` | `OnlineSecurity` | Indica si el cliente tiene contratado el servicio de seguridad en línea (`Yes`/`No`). | Es un servicio de valor agregado. Una mayor cantidad de servicios adicionales puede disminuir la probabilidad de cancelación. |
| `internet` | `OnlineBackup` | Indica si el cliente tiene contratado un servicio de respaldo en la nube (`Yes`/`No`). | Es un servicio de valor agregado que puede incrementar los costos de cambio para el cliente. |
| `internet` | `DeviceProtection` | Indica si el cliente cuenta con protección o seguro para sus dispositivos (`Yes`/`No`). | Refleja una mayor vinculación con la empresa y el interés del cliente por proteger su infraestructura. |
| `internet` | `TechSupport` | Indica si el cliente tiene contratado soporte técnico premium (`Yes`/`No`). | Variable fundamental. La falta de soporte ante fallas puede ser una de las causas de cancelación. |
| `internet` | `StreamingTV` | Indica si el cliente utiliza internet para transmitir contenido de televisión (`Yes`/`No`). | Representa un uso intensivo del ancho de banda. Los clientes insatisfechos con la velocidad podrían cancelar el servicio. |
| `internet` | `StreamingMovies` | Indica si el cliente utiliza internet para transmitir películas (`Yes`/`No`). | Variable relacionada con el entretenimiento y con la intensidad de uso del servicio de internet. |
| `phone` | `customerID` | Identificador único de cada cliente. | Permite identificar a los clientes que tienen contratado el servicio telefónico. |
| `phone` | `MultipleLines` | Indica si el cliente tiene múltiples líneas telefónicas (`Yes`/`No`). | Los clientes con varias líneas pueden tener una mayor dependencia del operador y facturas más elevadas. |

# 2.1. Fuentes de datos

El proyecto dispone de cuatro fuentes de información las cualales deben ser relacionadas por el campo `customerID`.

## `contract.csv`

Contiene información contractual y de facturación de los clientes.

### Variables principales

- `customerID`
- `BeginDate`
- `EndDate`
- `Type`
- `PaperlessBilling`
- `PaymentMethod`
- `MonthlyCharges`
- `TotalCharges`

Esta tabla contiene **7,043 clientes** y será utilizada como tabla principal del análisis.

## `personal.csv`

Contiene información personal de los clientes.

### Variables principales

- `customerID`
- `gender`
- `SeniorCitizen`
- `Partner`
- `Dependents`

Esta tabla contiene **7,043 clientes**.

## `internet.csv`

Contiene información sobre los servicios de internet contratados.

### Variables principales

- `customerID`
- `InternetService`
- `OnlineSecurity`
- `OnlineBackup`
- `DeviceProtection`
- `TechSupport`
- `StreamingTV`
- `StreamingMovies`

Esta tabla contiene **5,517 clientes**.

## `phone.csv`

Contiene información relacionada con el servicio telefónico.

### Variables principales

- `customerID`
- `MultipleLines`

Esta tabla contiene **6,361 clientes**.

## Integración de las fuentes

Las cuatro tablas se relacionarán mediante la variable `customerID`, que funciona como identificador único del cliente.

La tabla `contract.csv` será utilizada como tabla principal, y las tablas `personal.csv`, `internet.csv` y `phone.csv` se integrarán mediante operaciones de unión (*merge*), conservando la información disponible para cada cliente.

# 3. Plan de Trabajo Detallado: Metodología CRISP-DM

Propongo una estrategia estructurada en **4 pasos técnicos clave** para el d
esarrollo completo de la solución.


## Paso 1: Análisis Exploratorio de Datos (EDA) e Ingesta de Datos

### Carga de datos y Validación inicial de calidad de datos

Implementación de un pipeline de datos híbrido para conectar Python con una base de datos PostgreSQL local, manteniendo un respaldo (*fallback*) a archivos CSV locales para asegurar la portabilidad y reproducibilidad del entorno.

## Auditoría de calidad de los datos

Antes de realizar el análisis exploratorio de datos (EDA), se llevará a cabo una auditoría de calidad para evaluar la consistencia, integridad y confiabilidad de la información disponible.

Durante esta etapa se comprobarán los siguientes aspectos:

- Dimensiones de cada conjunto de datos.
- Nombres y estructura de las columnas.
- Tipos de datos de cada variable.
- Valores ausentes o nulos.
- Cadenas vacías y espacios en blanco.
- Valores categóricos inconsistentes.
- Registros duplicados.
- Identificadores `customerID` duplicados.
- Rangos y valores extremos de las variables numéricas.
- Coherencia y formato de las fechas.
- Integridad de las relaciones entre las tablas.

Esta auditoría permitirá detectar errores, inconsistencias o problemas estructurales antes de integrar los datos y comenzar el análisis exploratorio. De esta manera, se garantiza que las etapas posteriores de preparación, modelado y evaluación se realicen sobre información confiable.


### Análisis de clases

Evaluación visual del desequilibrio de clases en la variable objetivo `Churn` mediante gráficos de barras. También se analizará la relación entre los cargos mensuales (`MonthlyCharges`) y la tasa de abandono de clientes.

## Paso 2: Preparación de Datos y Feature Engineering

### Definición de la variable objetivo

Construcción de la etiqueta `Target` a partir de la columna `EndDate`:

- Si `EndDate` es igual a `"No"`, el cliente permanece activo y se asigna el valor `0`.
- Si `EndDate` contiene una fecha, significa que el cliente canceló y se asigna el valor `1`.

Para representar el evento empresarial que se desea predecir, se construirá la variable objetivo `Churn` de la siguiente manera:

- `Churn = 1`: cliente que canceló el servicio.
- `Churn = 0`: cliente que permanece activo.

La regla de construcción será:

```text
Churn = 1 cuando EndDate != "No"
Churn = 0 cuando EndDate == "No"
```

En términos conceptuales:

```python
Churn = (EndDate != "No").astype(int)
```

La variable original `EndDate` será retirada posteriormente del conjunto de características predictoras, ya que contiene directamente la información utilizada para definir el objetivo. Mantenerla durante el entrenamiento provocaría **fuga de información** (*data leakage*) y produciría una evaluación artificialmente optimista del modelo.

Esta definición deberá quedar explícitamente documentada, debido a que el enunciado proporciona una descripción ambigua de la característica objetivo. La documentación garantiza que la interpretación del evento de cancelación sea clara, reproducible y coherente durante todo el proyecto.
"""

### Creación de variables clave

Cálculo de la columna `Tenure`, que representa la antigüedad del cliente en días. Esta variable se obtiene restando la fecha de inicio (`BeginDate`) de la fecha de corte establecida:

```text
Fecha de corte: 1 de febrero de 2020
```

La fórmula conceptual es:

```text
Tenure = Fecha de corte - BeginDate
```

### Imputación y codificación

Gestión de los registros ausentes relacionados con los servicios telefónicos e internet, asignándoles la categoría `"No service"`.

Posteriormente, se aplicarán técnicas de codificación para las variables categóricas:

- **One-Hot Encoding** para modelos que requieren variables numéricas independientes.
- **Ordinal Encoding** cuando el algoritmo pueda beneficiarse de una representación ordenada de las categorías.

# 3. Análisis exploratorio de datos

El análisis exploratorio de datos (EDA) tendrá como objetivo comprender qué características parecen estar asociadas con la cancelación de los servicios.

El EDA no se limitará a la generación de gráficos. Cada análisis estará relacionado con una hipótesis de negocio que permita interpretar los resultados y orientar las siguientes etapas del proyecto.

## 3.1 Análisis univariado

En esta etapa se estudiarán los siguientes aspectos:

- Distribuciones de las variables numéricas.
- Categorías disponibles en las variables cualitativas.
- Frecuencias absolutas y relativas.
- Valores atípicos.
- Posibles errores de captura.
- Valores ausentes o inconsistentes.

### Variables numéricas iniciales

Las principales variables numéricas que se analizarán son:

- `MonthlyCharges`
- `TotalCharges`

Para estas variables se utilizarán medidas estadísticas descriptivas y representaciones gráficas, como histogramas y diagramas de caja.

### Variables categóricas

Se analizarán las siguientes variables categóricas:

- Tipo de contrato.
- Método de pago.
- Facturación electrónica.
- Características personales.
- Servicios contratados.
- Tipo de servicio de internet.
- Servicios adicionales.

Para cada variable se revisarán sus categorías, frecuencias y posibles inconsistencias en la escritura o representación de los valores.

## 3.2 Análisis del churn

Se comparará la tasa de *churn* entre distintos grupos de clientes, considerando las siguientes variables:

- Tipo de contrato.
- Método de pago.
- Nivel de cargos mensuales.
- Antigüedad del cliente.
- Servicio de internet.
- Servicio telefónico.
- Soporte técnico.
- Seguridad en línea.
- Servicio de respaldo (*backup*).
- Servicios de streaming.
- Características personales.

El objetivo será identificar patrones de comportamiento y generar hipótesis de negocio que posteriormente puedan ser verificadas mediante el modelo predictivo.

Por ejemplo, se analizará si los clientes con contratos mensuales, cargos elevados o poca antigüedad presentan una tasa de cancelación superior a la de otros segmentos.

## 4.1 Variable `Tenure`

La variable `Tenure` representará cuánto tiempo lleva el cliente relacionado con Interconnect.

### Justificación

La antigüedad puede representar estabilidad, fidelidad y grado de vinculación con la empresa.

La construcción de esta variable deberá evitar el uso de información futura o de datos disponibles únicamente después de que el cliente haya cancelado el servicio.

## 4.2 Indicadores de servicios

Se estudiará la creación de variables como:

- `HasInternet`
- `HasPhone`

### Justificación

La ausencia de un registro en las tablas de servicios puede contener información empresarial relevante. Por lo tanto, no debería interpretarse simplemente como un valor faltante.

## 4.3 Número de servicios contratados

Se podrá crear una característica denominada `ServiceCount`, que contabilice los servicios adicionales contratados por cada cliente.

### Hipótesis

Un cliente con varios productos puede presentar una mayor vinculación con la empresa y mayores costos de cambio frente a otra compañía.

Esta hipótesis será comprobada mediante el análisis exploratorio de datos.

## 4.4 Variables relacionadas con el precio

Se investigará la relación entre las siguientes variables:

- Cargos mensuales.
- Cargos totales.
- Antigüedad del cliente.
- Número de servicios contratados.

No se crearán ratios automáticamente. Únicamente se conservarán aquellas variables derivadas cuya interpretación y utilidad puedan justificarse desde el punto de vista del negocio.

# 5. Prevención de Data Leakage

La prevención de la fuga de información será uno de los criterios centrales del proyecto.

No podrán emplearse como predictores aquellas variables que revelen directamente que el cliente ya canceló el servicio.

Por esta razón, la variable `EndDate` no será utilizada como característica predictora después de construir la variable objetivo `Churn`.

También se evaluará cuidadosamente cualquier variable derivada de fechas para garantizar que pueda calcularse utilizando únicamente la información disponible en el momento en que se realizaría una predicción real.

# 6. Preparación de los datos para Machine Learning

Después del análisis exploratorio de datos, se realizará la preparación de la información para el modelado.

Las tareas previstas incluyen:

- Separación de la variable objetivo y los predictores.
- Eliminación de identificadores sin valor predictivo directo.
- Tratamiento de las variables categóricas.
- Transformación de variables numéricas cuando sea necesaria.
- Construcción de pipelines reproducibles.
- Separación de los datos de entrenamiento y evaluación.

La variable `customerID` se conservará para fines de trazabilidad, pero no será utilizada directamente como predictor del modelo.

# 7. Estrategia de separación de datos

El dataset se dividirá antes de realizar cualquier transformación que aprenda información estadística de los datos.

Se utilizará una partición estratificada para conservar aproximadamente la proporción de clientes con y sin *churn* en cada subconjunto.

Se considerará trabajar con los siguientes conjuntos:

- Conjunto de entrenamiento.
- Conjunto de validación o validación cruzada.
- Conjunto final de prueba.

El conjunto de prueba permanecerá aislado durante toda la experimentación y se utilizará únicamente para evaluar el modelo seleccionado.

# 8. Modelo baseline

Antes de probar algoritmos complejos, se establecerán modelos de referencia.

## `DummyClassifier`

El modelo `DummyClassifier` permitirá medir qué rendimiento puede obtenerse sin aprender relaciones reales entre las variables.

## Regresión logística

La Regresión Logística será utilizada como baseline interpretable.

Este modelo permitirá:

- Establecer una referencia más competitiva.
- Identificar relaciones iniciales entre las variables y el *churn*.
- Comparar modelos más complejos con una alternativa sencilla.

Un modelo complejo solo se considerará superior si demuestra una mejora real con respecto a estos modelos baseline.

# 9. Modelos candidatos

Después de establecer los modelos baseline, se evaluarán diferentes familias de algoritmos.

Entre los posibles candidatos se incluyen:

- `LogisticRegression`
- `RandomForest`
- `GradientBoosting`
- `CatBoost`
- Otros algoritmos de *boosting* disponibles en el entorno.

La selección no se realizará anticipadamente. Cada modelo será comparado utilizando la misma estrategia de validación.

`CatBoost` será un candidato especialmente interesante debido a la presencia de múltiples características categóricas. Sin embargo, su utilización deberá justificarse empíricamente a partir del rendimiento obtenido.

# 10. Optimización de hiperparámetros

La búsqueda de hiperparámetros se realizará únicamente después de identificar los modelos más prometedores.

No se optimizarán indiscriminadamente todos los algoritmos.

El proceso seguirá las siguientes etapas:

```text
Baseline → Comparación → Selección de candidatos → Tuning
```

Esta estrategia permitirá reducir la complejidad experimental y mantener la trazabilidad sobre las mejoras obtenidas.

# 11. Métricas de evaluación

La métrica principal será el **AUC-ROC**.

Esta métrica se utilizará porque evalúa la capacidad del modelo para ordenar correctamente a los clientes según sus diferentes niveles de riesgo, sin depender de un único umbral de clasificación.

Esto resulta especialmente relevante cuando el objetivo es priorizar clientes para campañas de retención.

## Objetivo técnico

El objetivo técnico será alcanzar:

```text
AUC-ROC ≥ 0.88
```

Este valor permitirá aspirar al nivel máximo establecido por el criterio de evaluación del proyecto.

## Accuracy

La exactitud (*accuracy*) se reportará como métrica adicional, conforme a los requisitos del proyecto.

Sin embargo, no será interpretada de forma aislada debido al posible desbalance existente entre clientes activos y clientes que cancelan.

## Métricas complementarias

Para realizar un análisis más completo y profesional, también se considerarán las siguientes métricas:

- `Precision`
- `Recall`
- `F1-score`
- Matriz de confusión
- `PR-AUC`

Estas métricas permitirán comprender mejor el comportamiento operativo del modelo.

# 12. Selección del umbral

El umbral de clasificación no tendrá que permanecer necesariamente en `0.50`.

Una vez seleccionado el modelo, se evaluará el compromiso entre:

- Clientes en riesgo detectados.
- Falsos positivos.
- Capacidad operativa del área de Marketing.

Por ejemplo, si el área de Marketing únicamente puede contactar a una fracción determinada de la cartera, las probabilidades generadas por el modelo podrán utilizarse para ordenar a los clientes y seleccionar aquellos con mayor riesgo de abandono.

# 18. Interpretabilidad

El proyecto no finalizará únicamente con una métrica de rendimiento.

También se analizará qué variables están influyendo en las predicciones del modelo.

Dependiendo del algoritmo seleccionado, podrán utilizarse las siguientes técnicas:

- Coeficientes del modelo.
- `Feature importance`.
- `Permutation importance`.
- SHAP.
- Otras técnicas de interpretabilidad apropiadas.

El objetivo será traducir la información del modelo en conclusiones comprensibles para el área de negocio.

Se evitará interpretar automáticamente la importancia predictiva como causalidad.

# 13. Traducción de resultados al negocio

La solución final deberá responder preguntas como:

- ¿Qué perfiles presentan mayor riesgo de abandono?
- ¿Qué variables ayudan más a distinguir a los clientes en riesgo?
- ¿En qué momento de la relación con el cliente parece existir una mayor vulnerabilidad?
- ¿Qué segmentos deberían ser priorizados?
- ¿Cuántos clientes podría manejar una campaña utilizando diferentes umbrales?
- ¿Qué limitaciones tiene la predicción?

Las recomendaciones estarán basadas en la evidencia obtenida durante el análisis y no únicamente en intuiciones.

# 14. Flujo general del proyecto

El proyecto seguirá un flujo estructurado basado en las etapas de la metodología CRISP-DM.

## Orden de ejecución

1. Comprensión del problema de negocio.
2. Auditoría de los datasets.
3. Integración de las fuentes de datos.
4. Limpieza y transformación inicial.
5. Definición de la variable objetivo.
6. Análisis exploratorio de datos.
7. Ingeniería de características.
8. Separación de los datos.
9. Construcción del pipeline.
10. Entrenamiento de modelos baseline.
11. Comparación de algoritmos.
12. Optimización de hiperparámetros.
13. Evaluación final del modelo.
14. Análisis de interpretabilidad.
15. Elaboración de conclusiones y recomendaciones de negocio.

Este flujo permitirá desarrollar la solución de forma ordenada, reproducible y alineada con los objetivos empresariales de Interconnect.

# 15. Criterio de éxito

El proyecto se considerará técnicamente satisfactorio cuando cumpla con los siguientes criterios:

- El pipeline pueda reproducirse de forma consistente.
- No exista fuga de información (*data leakage*).
- El modelo supere claramente el rendimiento de los modelos baseline.
- El modelo sea evaluado sobre datos que no hayan sido utilizados durante el entrenamiento.
- Se alcance un desempeño competitivo en la métrica AUC-ROC.
- Las métricas de evaluación sean interpretadas correctamente.
- Los resultados puedan traducirse en una estrategia de priorización de clientes.

## Éxito desde el punto de vista empresarial

Desde el punto de vista empresarial, el éxito dependerá de que las probabilidades producidas por el modelo puedan utilizarse para identificar y priorizar a los clientes con mayor riesgo de cancelación antes de que abandonen Interconnect.

De esta manera, el área de Marketing podrá enfocar sus esfuerzos en los clientes más vulnerables y diseñar acciones de retención, como promociones, descuentos, planes especiales o atención personalizada.

# 2 Auditoría de los datasets.

# Interconnect — Predicción de Churn
## 01. Validación y preparación inicial de datos

### Objetivo

En este notebook se realizará la validación inicial de las cuatro fuentes de datos proporcionadas por Interconnect:

- `contract.csv`
- `personal.csv`
- `internet.csv`
- `phone.csv`

El objetivo de esta etapa es evaluar la estructura, calidad e integridad de los datos antes de realizar transformaciones, análisis exploratorio o entrenamiento de modelos.

Se revisarán:

- dimensiones de los datasets;
- tipos de datos;
- valores ausentes;
- cadenas vacías;
- registros duplicados;
- unicidad de `customerID`;
- valores categóricos;
- consistencia entre las diferentes fuentes.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

**Definir la ubicación de los datasets**

In [4]:
DATA_DIR = Path("/Volumes/PortableSSD/TripleTen/CDD/Sprint_19/proyecto_interconnect/data/final_provider")

# Cargar los cuatro datasets

contract = pd.read_csv(DATA_DIR / "contract.csv")
personal = pd.read_csv(DATA_DIR / "personal.csv")
internet = pd.read_csv(DATA_DIR / "internet.csv")
phone = pd.read_csv(DATA_DIR / "phone.csv")

In [5]:
# diccionario será muy útil. Con el podremos recorrer los cuatro datasets.
datasets = {
    "contract": contract,
    "personal": personal,
    "internet": internet,
    "phone": phone
}

In [6]:
# Verificar que la carga fue correcta

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

contract: (7043, 8)
personal: (7043, 5)
internet: (5517, 8)
phone: (6361, 2)


**Interpretación**

contract y personal contienen:

7,043 clientes

pero:

internet → 5,517
phone    → 6,361

Esto no significa automáticamente que falten datos.

Según la descripción oficial, Interconnect ofrece teléfono e Internet como servicios distintos y un cliente puede contratar diferentes servicios.

Por eso nuestra primera hipótesis es que la ausencia de ciertos clientes en estas tablas representa servicio no contratado.

**Inspección inicial**

In [7]:
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(name.upper())
    print(f"{'=' * 60}")
    
    display(df.head())


CONTRACT


,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,1889.5
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,Month-to-month,Yes,Mailed check,53.85,108.15
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,70.70,151.65



PERSONAL


,customerID,gender,SeniorCitizen,Partner,Dependents
0,7590-VHVEG,Female,0,Yes,No
1,5575-GNVDE,Male,0,No,No
2,3668-QPYBK,Male,0,No,No
3,7795-CFOCW,Male,0,No,No
4,9237-HQITU,Female,0,No,No



INTERNET


,customerID,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,7590-VHVEG,DSL,No,Yes,No,No,No,No
1,5575-GNVDE,DSL,Yes,No,Yes,No,No,No
2,3668-QPYBK,DSL,Yes,Yes,No,No,No,No
3,7795-CFOCW,DSL,Yes,No,Yes,Yes,No,No
4,9237-HQITU,Fiber optic,No,No,No,No,No,No



PHONE


,customerID,MultipleLines
0,5575-GNVDE,No
1,3668-QPYBK,No
2,9237-HQITU,No
3,9305-CDSKC,Yes
4,1452-KIOVK,Yes


In [8]:
# Revisar tipos de datos

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.dtypes)


CONTRACT
customerID           object
BeginDate            object
EndDate              object
Type                 object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
dtype: object

PERSONAL
customerID       object
gender           object
SeniorCitizen     int64
Partner          object
Dependents       object
dtype: object

INTERNET
customerID          object
InternetService     object
OnlineSecurity      object
OnlineBackup        object
DeviceProtection    object
TechSupport         object
StreamingTV         object
StreamingMovies     object
dtype: object

PHONE
customerID       object
MultipleLines    object
dtype: object


## ¿Qué esperamos conceptualmente?

Durante la auditoría de tipos de datos se identificó que algunas columnas no tienen el formato más adecuado para el análisis.

### `BeginDate`

Actualmente, la columna tiene el tipo:

```text
object
```

Sin embargo, conceptualmente debería representar una fecha. Por lo tanto, será necesario convertirla a un tipo de dato compatible con fechas, como `datetime`.

### `EndDate`

La columna también tiene actualmente el tipo:

```text
object
```

Este comportamiento es esperado porque contiene una combinación de:

- El valor `"No"`, que indica que el cliente continúa activo.
- Fechas, que indican que el cliente canceló el servicio.

Será necesario tratar esta columna cuidadosamente para diferenciar los clientes activos de los clientes que cancelaron.

### `TotalCharges`

Actualmente, la columna tiene el tipo:

```text
object
```

Sin embargo, conceptualmente debería ser una variable numérica de tipo:

```text
float
```

Esta conversión permitirá realizar cálculos estadísticos, analizar distribuciones y utilizar la variable en los modelos de Machine Learning.

## Primera inconsistencia técnica

La columna `TotalCharges` representa la primera inconsistencia técnica real detectada durante la auditoría, ya que contiene información numérica almacenada como texto.

Además, las columnas `BeginDate` y `EndDate` deberán convertirse o procesarse adecuadamente para trabajar con fechas y evitar errores durante la creación de variables como `Tenure` y `Churn`.

## Tabla resumen

In [9]:
summary = pd.DataFrame({
    "rows": {name: df.shape[0] for name, df in datasets.items()},
    "columns": {name: df.shape[1] for name, df in datasets.items()},
    "duplicated_rows": {
        name: df.duplicated().sum()
        for name, df in datasets.items()
    },
    "duplicated_customerID": {
        name: df["customerID"].duplicated().sum()
        for name, df in datasets.items()
    }
})

summary

,rows,columns,duplicated_rows,duplicated_customerID
contract,7043,8,0,0
personal,7043,5,0,0
internet,5517,8,0,0
phone,6361,2,0,0


Tenemos:

duplicated_rows = 0 duplicated_customerID = 0

Es una buena señal y nos permite tomar la variable customerID como lave primaria dentro de cada archivo.

### Validar formalmente la unicidad de customerID

In [10]:
for name, df in datasets.items():
    assert df["customerID"].is_unique, (    # Por qué usar assert. estamos convirtiendo una observación en una regla de calidad. Si mañana recibimos otros datos y aparecen IDs repetidos, el notebook se detendrá.
        f"customerID no es único en {name}"
    )

print("customerID es único en todos los datasets.")

customerID es único en todos los datasets.


### Revisar valores nulos tradicionales

In [11]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    
    missing = (
        df.isna()
        .sum()
        .sort_values(ascending=False)
    )
    
    print(missing)


CONTRACT
customerID          0
BeginDate           0
EndDate             0
Type                0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
dtype: int64

PERSONAL
customerID       0
gender           0
SeniorCitizen    0
Partner          0
Dependents       0
dtype: int64

INTERNET
customerID          0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
dtype: int64

PHONE
customerID       0
MultipleLines    0
dtype: int64


**0 valores NaN no significa necesariamente 0 problemas de datos.**

* Porque ya sabemos que TotalCharges tiene valores vacíos representados como espacios.

* Pandas no los detectó como NaN.

### Buscar cadenas vacías ocultas

In [12]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    
    object_columns = df.select_dtypes(include=["object", "string"]).columns

    for column in object_columns:
        blank_count = (
            df[column]
            .astype(str)
            .str.strip() # Elimina espacios al principio y al final. un valor como: " " pasa temporalmente a: ""
            .eq("") # comprueba si quedó vacío.
            .sum()
        )
        
        if blank_count > 0:
            print(f"{column}: {blank_count} cadenas vacías")


CONTRACT
TotalCharges: 11 cadenas vacías

PERSONAL

INTERNET

PHONE


### Inspeccionar esos 11 registros

In [13]:
blank_total_charges = (
    contract["TotalCharges"]
    .astype(str)
    .str.strip()
    .eq("")
)

contract.loc[
    blank_total_charges,
    [
        "customerID",
        "BeginDate",
        "EndDate",
        "MonthlyCharges",
        "TotalCharges"
    ]
]

,customerID,BeginDate,EndDate,MonthlyCharges,TotalCharges
488,4472-LVYGI,2020-02-01,No,52.55,
753,3115-CZMZD,2020-02-01,No,20.25,
936,5709-LVOEQ,2020-02-01,No,80.85,
1082,4367-NUYAO,2020-02-01,No,25.75,
1340,1371-DWPAZ,2020-02-01,No,56.05,
3331,7644-OMVMY,2020-02-01,No,19.85,
3826,3213-VVOLG,2020-02-01,No,25.35,
4380,2520-SGTTA,2020-02-01,No,20.00,
5218,2923-ARZLG,2020-02-01,No,19.70,
6670,4075-WKNIU,2020-02-01,No,73.35,


**Las instrucciones indican que la información contractual es válida a partir del 1 de febrero de 2020.**

hipótesis fuerte:

Estos 11 clientes probablemente fueron dados de alta exactamente en la fecha de corte y todavía no habían acumulado un histórico de TotalCharges.

### Revisar las categorías existentes

In [14]:
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(name.upper())
    print(f"{'=' * 60}")
    
    categorical_columns = [
        col for col in df.select_dtypes(include=["object", "string"]).columns
        if col != "customerID"
    ]
    
    for column in categorical_columns:
        print(f"\n{column}")
        print(df[column].value_counts(dropna=False))


CONTRACT

BeginDate
BeginDate
2014-02-01    366
2019-10-01    237
2019-11-01    237
2019-09-01    237
2020-01-01    233
2019-12-01    220
2014-03-01    178
2019-07-01    156
2019-08-01    146
2019-06-01    141
2019-02-01    128
2019-05-01    123
2019-01-01    114
2014-04-01    114
2019-03-01    110
2019-04-01    108
2018-08-01    100
2018-09-01    100
2018-11-01     99
2014-07-01     98
2014-05-01     98
2014-06-01     97
2017-03-01     94
2018-02-01     91
2017-12-01     90
2018-12-01     90
2018-03-01     89
2018-06-01     84
2014-08-01     82
2017-11-01     82
2015-10-01     81
2018-04-01     81
2014-10-01     80
2015-02-01     80
2018-07-01     80
2015-06-01     79
2018-10-01     78
2014-11-01     77
2014-12-01     75
2015-01-01     75
2016-09-01     74
2015-05-01     74
2015-11-01     74
2016-02-01     73
2017-09-01     73
2018-01-01     73
2016-04-01     73
2014-09-01     72
2017-08-01     71
2015-04-01     69
2016-05-01     68
2015-12-01     68
2015-09-01     67
2016-08-01     

### Categorías de Internet

Nuestros datos reales muestran:

**InternetService**

* Fiber optic    3096
* DSL            2421


Y variables como:   

- OnlineSecurity
- OnlineBackup
- DeviceProtection
- TechSupport
- StreamingTV
- StreamingMovies

solo contienen: Yes, No

Esto es importante porque en internet.csv no aparece:

No internet service como categoría explícita.

La ausencia del cliente en esa tabla es precisamente la que probablemente representa esa situación.

Más adelante tendremos que reconstruirla durante el merge.

### Telefonía

## Análisis de la variable `MultipleLines`

La variable `MultipleLines` contiene las siguientes categorías:

| Categoría | Frecuencia |
|---|---:|
| `No` | 3,390 |
| `Yes` | 2,971 |

Esto indica que, dentro de la tabla `phone`, existen clientes con una línea telefónica y clientes con múltiples líneas.

## Ausencia de la categoría `No phone service`

En esta tabla no existe la categoría:

```text
No phone service
```

Esto ocurre porque los clientes que no tienen contratado el servicio telefónico simplemente no aparecen en `phone.csv`.

Éste será otro caso donde un `NaN` después del merge tendrá significado empresarial.

### Revisión de la variable `SeniorCitizen`

Existe un aspecto sutil relacionado con la variable `SeniorCitizen`.

### Tipo de almacenamiento

Pandas detecta esta variable con el tipo:

```text
SeniorCitizen → int64
```

Esto ocurre porque sus valores están representados mediante números enteros:

```text
0
1
```

### Significado estadístico

Aunque la variable se almacena como numérica, desde el punto de vista estadístico no representa una variable numérica continua.

No tendría sentido interpretar:

```text
SeniorCitizen = 1
```

como si fuera "el doble" de:

```text
SeniorCitizen = 0
```

Los valores representan categorías:

- `0`: el cliente no es adulto mayor.
- `1`: el cliente es adulto mayor.

Por lo tanto, `SeniorCitizen` es una variable binaria o categórica codificada numéricamente.

## Tratamiento posterior

Durante la preparación de los datos, esta variable será tratada como categórica o binaria, según el pipeline y el algoritmo seleccionado.

Este caso demuestra una distinción importante:

> **El tipo de almacenamiento no siempre coincide con el significado estadístico de una variable.**

Que una columna esté almacenada como `int64` no significa necesariamente que deba tratarse como una variable numérica continua.

### Validar IDs entre tablas

In [15]:
contract_ids = set(contract["customerID"])
personal_ids = set(personal["customerID"])
internet_ids = set(internet["customerID"])
phone_ids = set(phone["customerID"])

In [16]:
len(contract_ids - personal_ids)

0

In [17]:
len(personal_ids - contract_ids)

0

Es decir: contract ↔ personal tienen exactamente los mismos clientes.

Muy buena señal.

### Clientes sin registro de Internet

In [18]:
customers_without_internet = contract_ids - internet_ids

len(customers_without_internet)

1526

In [19]:
len(customers_without_internet) / len(contract_ids)

0.21666903308249325

Es decir: 21.7 % de los clientes no aparecen en internet.csv.

No diremos todavía que son missing values.

Diremos:

1,526 clientes de la cartera contractual no tienen un registro correspondiente en la tabla de Internet.

Después interpretaremos el significado.


### Clientes sin registro telefónico

In [20]:
customers_without_phone = contract_ids - phone_ids

len(customers_without_phone)

682

In [21]:
len(customers_without_phone) / len(contract_ids)

0.09683373562402385

Es decir: 9.7 % de los clientes no aparecen en phone.csv

No diremos todavía que son missing values.

Diremos:

682 clientes de la cartera contractual no tienen un registro correspondiente en la tabla de phone.

Después interpretaremos el significado.

### Tabla de integridad entre fuentes

In [22]:
relationship_summary = pd.DataFrame({
    "dataset": ["personal", "internet", "phone"],
    "customers": [
        len(personal_ids),
        len(internet_ids),
        len(phone_ids)
    ],
    "missing_vs_contract": [
        len(contract_ids - personal_ids),
        len(contract_ids - internet_ids),
        len(contract_ids - phone_ids)
    ]
})

relationship_summary

,dataset,customers,missing_vs_contract
0,personal,7043,0
1,internet,5517,1526
2,phone,6361,682


### Conclusiones de la auditoría inicial

La revisión estructural muestra que los cuatro datasets utilizan `customerID` como identificador único y no presentan registros duplicados.

`contract.csv` y `personal.csv` contienen la misma población de 7,043 clientes.

Por su parte, `internet.csv` y `phone.csv` contienen menos registros. Esto no se interpretará inicialmente como pérdida de información, ya que estas tablas representan servicios específicos y un cliente puede no haberlos contratado.

Se detectó una inconsistencia en `TotalCharges`: la variable está almacenada como texto y contiene 11 cadenas vacías. Todos estos registros pertenecen a clientes cuya fecha de inicio es 2020-02-01, correspondiente a la fecha de referencia de los datos, lo que sugiere que podrían ser clientes recién incorporados sin cargos acumulados.

También será necesario convertir adecuadamente las variables temporales `BeginDate` y `EndDate` antes de utilizarlas en análisis posteriores.

# 3. Integración de las fuentes

El objetivo de esta fase es construir un único DataFrame con los 7,043 clientes, combinando la información contractual, personal, de Internet y de telefonía, sin perder clientes y sin duplicarlos.

## Arquitectura de integración de datos

La tabla principal del proyecto será `contract`, ya que contiene la información contractual, las fechas y los cargos de los clientes.

Las demás tablas se integrarán mediante operaciones `LEFT JOIN` utilizando la columna `customerID` como clave de unión:

```text
contract
│
├── LEFT JOIN personal
│
├── LEFT JOIN internet
│
└── LEFT JOIN phone
```

## Justificación del uso de `LEFT JOIN`

El uso de `LEFT JOIN` permite conservar todos los clientes presentes en la tabla `contract`, aunque no tengan registros asociados en las tablas de servicios.

La estructura de integración será equivalente a:

```sql
contract
    LEFT JOIN personal
        ON contract.customerID = personal.customerID
    LEFT JOIN internet
        ON contract.customerID = internet.customerID
    LEFT JOIN phone
        ON contract.customerID = phone.customerID
```

De esta forma:

- `contract` funcionará como tabla principal.
- `personal` aportará las características personales.
- `internet` aportará la información de los servicios de internet.
- `phone` aportará la información del servicio telefónico.
- Los clientes sin servicios de internet o telefonía no serán eliminados.
- Los valores ausentes resultantes del `LEFT JOIN` podrán interpretarse posteriormente como `"No internet service"` o `"No phone service"`, según corresponda.

### Antes de hacer cualquier merge()

Registro de cuantas filas tenemos.

In [23]:
n_customers_before = contract.shape[0]

print(f"Clientes antes de integrar: {n_customers_before}")

Clientes antes de integrar: 7043


In [24]:
print(
    "customerID únicos:",
    contract["customerID"].nunique()
)

customerID únicos: 7043


Nuestra condición inicial es entonces:

filas = clientes unicos=7043

#### Primer merge: contract + personal

In [25]:
data = contract.merge(
    personal,
    on="customerID", # utiliza customerID como llave de unión.
    how="left", # conserva todos los clientes de contract.
    validate="one_to_one" # espero que cada customerID aparezca una sola vez en contract y una sola vez en personal
)

# Validar el primer merge

print("Shape:", data.shape)
print("customerID únicos:", data["customerID"].nunique())
print("Duplicados:", data["customerID"].duplicated().sum())

Shape: (7043, 12)
customerID únicos: 7043
Duplicados: 0


#### Segundo merge: internet

In [26]:
data = data.merge(
    internet,
    on="customerID",
    how="left",
    validate="one_to_one"
)

# Validar el segundo merge

print("Shape:", data.shape)
print("customerID únicos:", data["customerID"].nunique())
print("Duplicados:", data["customerID"].duplicated().sum())

Shape: (7043, 19)
customerID únicos: 7043
Duplicados: 0


#### ¿Cuántos NaN deberían aparecer?

In [27]:
data["InternetService"].isna().sum()

np.int64(1526)

* contract.csv  → 7043
* internet.csv  → 5517


7043 − 5517 = 1526

#### Validar que todas las columnas de Internet coincidan

In [28]:
internet_columns = [
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

data[internet_columns].isna().sum()

InternetService     1526
OnlineSecurity      1526
OnlineBackup        1526
DeviceProtection    1526
TechSupport         1526
StreamingTV         1526
StreamingMovies     1526
dtype: int64

No faltan valores aislados al azar; falta todo el bloque de Internet para ciertos clientes.

#### Tercer merge: phone

In [29]:
data = data.merge(
    phone,
    on="customerID",
    how="left",
    validate="one_to_one"
)

# Validar el tercer merge

print("Shape:", data.shape)
print("customerID únicos:", data["customerID"].nunique())
print("Duplicados:", data["customerID"].duplicated().sum()) 

Shape: (7043, 20)
customerID únicos: 7043
Duplicados: 0


#### Comprobar telefonía

In [30]:
data["MultipleLines"].isna().sum()

np.int64(682)

* contract.csv  → 7043
* phone.csv  → 6361

7043 − 6361 = 682

#### Validación final de la integración

In [31]:
assert data.shape[0] == n_customers_before, (
    "Se perdió o duplicó algún cliente durante la integración."
)

assert data["customerID"].nunique() == n_customers_before, (
    "El número de customerID únicos cambió."
)

assert data["customerID"].is_unique, (
    "Existen customerID duplicados después de la integración."
)

print("Integración completada correctamente.")
print(f"Clientes finales: {data.shape[0]}")
print(f"Variables finales: {data.shape[1]}")

Integración completada correctamente.
Clientes finales: 7043
Variables finales: 20


## Comprobación del número de columnas

Antes de integrar las tablas, se puede calcular cuántas columnas tendrá el dataset final.

### Tabla `contract`

La tabla `contract` contiene inicialmente:

```text
8 columnas
```

### Tabla `personal`

La tabla `personal` contiene 5 columnas, pero `customerID` ya existe en `contract`.

Por lo tanto, únicamente se añadirán:

```text
4 columnas
```

El total acumulado será:

```text
8 + 4 = 12 columnas
```

### Tabla `internet`

La tabla `internet` contiene 8 columnas, pero `customerID` ya existe en la tabla integrada.

Por lo tanto, se añadirán:

```text
7 columnas
```

El total acumulado será:

```text
12 + 7 = 19 columnas
```

### Tabla `phone`

La tabla `phone` contiene 2 columnas, pero `customerID` ya existe.

Por lo tanto, se añadirá:

```text
1 columna
```

El total final será:

```text
19 + 1 = 20 columnas
```

## Forma esperada del dataset final

La tabla integrada debería tener:

```text
7,043 filas × 20 columnas
```

En notación de Python:

```python
(7043, 20)
```

Esta expectativa servirá como comprobación de consistencia después de realizar los `LEFT JOIN`.

### Revisar el resultado

In [32]:
data.head()

,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,gender,SeniorCitizen,Partner,Dependents,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,MultipleLines
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,29.85,Female,0,Yes,No,DSL,No,Yes,No,No,No,No,NaN
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,1889.5,Male,0,No,No,DSL,Yes,No,Yes,No,No,No,No
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,Month-to-month,Yes,Mailed check,53.85,108.15,Male,0,No,No,DSL,Yes,Yes,No,No,No,No,No
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1840.75,Male,0,No,No,DSL,Yes,No,Yes,Yes,No,No,NaN
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,70.70,151.65,Female,0,No,No,Fiber optic,No,No,No,No,No,No,No


In [33]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   BeginDate         7043 non-null   object 
 2   EndDate           7043 non-null   object 
 3   Type              7043 non-null   object 
 4   PaperlessBilling  7043 non-null   object 
 5   PaymentMethod     7043 non-null   object 
 6   MonthlyCharges    7043 non-null   float64
 7   TotalCharges      7043 non-null   object 
 8   gender            7043 non-null   object 
 9   SeniorCitizen     7043 non-null   int64  
 10  Partner           7043 non-null   object 
 11  Dependents        7043 non-null   object 
 12  InternetService   5517 non-null   object 
 13  OnlineSecurity    5517 non-null   object 
 14  OnlineBackup      5517 non-null   object 
 15  DeviceProtection  5517 non-null   object 
 16  TechSupport       5517 non-null   object 


Una fila representa:

un cliente de Interconnect.

### Crear un resumen de integración

In [34]:
integration_summary = pd.DataFrame({
    "metric": [
        "Clientes originales",
        "Clientes finales",
        "customerID únicos",
        "customerID duplicados",
        "Sin registro Internet",
        "Sin registro Phone"
    ],
    "value": [
        n_customers_before,
        data.shape[0],
        data["customerID"].nunique(),
        data["customerID"].duplicated().sum(),
        data["InternetService"].isna().sum(),
        data["MultipleLines"].isna().sum()
    ]
})

integration_summary

,metric,value
0,Clientes originales,7043
1,Clientes finales,7043
2,customerID únicos,7043
3,customerID duplicados,0
4,Sin registro Internet,1526
5,Sin registro Phone,682


In [35]:
print(data.columns.tolist())

['customerID', 'BeginDate', 'EndDate', 'Type', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'MultipleLines']


### Conclusiones de la integración

Las cuatro fuentes de información fueron integradas utilizando `customerID` como llave mediante uniones de tipo `LEFT JOIN`, tomando `contract.csv` como tabla principal.

La integración conservó los 7,043 clientes originales y no generó identificadores duplicados, por lo que se mantiene una relación de una fila por cliente.

Después de integrar `internet.csv`, se observaron 1,526 clientes sin información asociada a servicios de Internet. Asimismo, después de integrar `phone.csv`, 682 clientes no presentan información telefónica.

Estas ausencias son consistentes con las diferencias detectadas previamente entre las fuentes y no serán imputadas en esta etapa. Su significado será tratado durante la fase de limpieza y transformación inicial.

## Guardar el dataset unificado que servirá de entrada para el Cuaderno 2:

In [36]:
# Guardamos el dataset crudo integrado en la carpeta data
data.to_csv('../data/interconnect_raw.csv', index=False)
print("¡Dataset crudo integrado guardado con éxito en la carpeta data!")


¡Dataset crudo integrado guardado con éxito en la carpeta data!
